[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limorigu/uchicago-python-ai-workshop/blob/main/notebooks/01b_excel_to_pandas_demo.ipynb)

# 01b · Excel → pandas, side by side (live demo)

We step through this together in the Day 1 pandas talk — keep it open and run each cell as it appears on screen. It shows the **two Excel moves analysts use most** — the pivot table and the VLOOKUP — built the Excel way (described) and the pandas way (run live), *including the ways each one fails*.

We use a small, readable dataset: the five-position book from Lab B, plus a reference table — small enough to check every number by eye. That is the point: learn the move where you can verify it, then apply it to 236,549 rows in Lab C.

In [1]:
import pandas as pd
import numpy as np

trades = pd.DataFrame({
    "ticker":  ["HLT", "HLT", "MAR", "MAR", "H",  "CHH", "WH", "WH", "WH"],
    "desk":    ["Lodging", "Lodging", "Lodging", "Lodging", "Lodging", "Franchise", "Franchise", "Franchise", "Franchise"],
    "side":    ["BUY", "SELL", "BUY", "BUY", "BUY", "BUY", "BUY", "SELL", "BUY"],
    "shares":  [40_000, 10_000, 25_000, 15_000, 32_000, 28_000, 51_000, 11_000, 20_000],
    "price":   [61.40, 62.10, 155.20, 156.00, 98.75, 118.30, 72.15, 73.00, 72.60],
})
trades["notional"] = trades.shares * trades.price
trades

,ticker,desk,side,shares,price,notional
0,HLT,Lodging,BUY,40000,61.40,2456000.0
1,HLT,Lodging,SELL,10000,62.10,621000.0
2,MAR,Lodging,BUY,25000,155.20,3880000.0
3,MAR,Lodging,BUY,15000,156.00,2340000.0
4,H,Lodging,BUY,32000,98.75,3160000.0
5,CHH,Franchise,BUY,28000,118.30,3312400.0
6,WH,Franchise,BUY,51000,72.15,3679650.0
7,WH,Franchise,SELL,11000,73.00,803000.0
8,WH,Franchise,BUY,20000,72.60,1452000.0


## 1 · The pivot table

**In Excel:** select the data → Insert → PivotTable → drag `ticker` to Rows, `side` to Columns, `notional` to Values → change "Count" to "Sum" (everyone forgets this once).

**In pandas:** one line, and it is *text* — you can read it in six months, diff it, and re-run it.

In [6]:
pivot = trades.pivot_table(index="ticker", columns="side", values="notional", aggfunc="sum", fill_value=0)
pivot

side,BUY,SELL
ticker,,
CHH,3312400.0,0.0
H,3160000.0,0.0
HLT,2456000.0,621000.0
MAR,6220000.0,0.0
WH,5131650.0,803000.0


In [7]:
# groupby is the more general tool — most 'pivots' in practice are groupbys:
summary = trades.groupby("ticker").agg(
    trades_n=("ticker", "count"),
    total_shares=("shares", "sum"),
    gross_notional=("notional", "sum"),
    avg_price=("price", "mean"),
)
summary

,trades_n,total_shares,gross_notional,avg_price
ticker,,,,
CHH,1,28000,3312400.0,118.300000
H,1,32000,3160000.0,98.750000
HLT,2,50000,3077000.0,61.750000
MAR,2,40000,6220000.0,155.600000
WH,3,82000,5934650.0,72.583333


In [8]:
# Multi-level pivots — Rows: desk AND ticker. In Excel: drag two fields. Here: a list.
trades.groupby(["desk", "ticker"]).notional.sum()

desk       ticker
Franchise  CHH       3312400.0
           WH        5934650.0
Lodging    H         3160000.0
           HLT       3077000.0
           MAR       6220000.0
Name: notional, dtype: float64

## 2 · The VLOOKUP

Attach each ticker's sector and daily vol from a reference table.

**In Excel:** `=XLOOKUP(A2, ref!A:A, ref!B:B)` (or the classic `=VLOOKUP(A2, ref!A:C, 2, FALSE)` — exact match flag, column counted by hand).

**In pandas:** `merge` — a left join.

In [12]:
ref.head()

,ticker,sector,daily_vol
0,HLT,Lodging,0.018
1,MAR,Lodging,0.016
2,H,Lodging,0.022
3,CHH,Franchise,0.019


In [14]:
ref = pd.DataFrame({
    "ticker": ["HLT", "MAR", "H", "CHH"],              # note: WH is missing — deliberately
    "sector": ["Lodging", "Lodging", "Lodging", "Franchise"],
    "daily_vol": [0.018, 0.016, 0.022, 0.019],
})
looked_up = summary.reset_index().merge(ref, on="ticker", how="left")
looked_up

,ticker,trades_n,total_shares,gross_notional,avg_price,sector,daily_vol
0,CHH,1,28000,3312400.0,118.300000,Franchise,0.019
1,H,1,32000,3160000.0,98.750000,Lodging,0.022
2,HLT,2,50000,3077000.0,61.750000,Lodging,0.018
3,MAR,2,40000,6220000.0,155.600000,Lodging,0.016
4,WH,3,82000,5934650.0,72.583333,NaN,NaN


### Failure case 1 — the key is missing
Excel shows `#N/A`. pandas shows `NaN`. Same disease, different rash. **Always count your NaNs after a merge:**

In [18]:
missing = looked_up[looked_up.sector.isna()]
print(f"{len(missing)} ticker(s) missing from the reference table: {list(missing.ticker)}")
# Fix: add the row to ref (best), or .fillna() with an explicit default (document why).

1 ticker(s) missing from the reference table: ['WH']


### Failure case 2 — the key is duplicated on the right side
The quiet killer. **Excel's VLOOKUP silently returns the *first* match** — no warning, wrong numbers forever. pandas does something louder: it **multiplies the rows**. Louder is better: you can test for it.

In [24]:
ref_dup = pd.concat([ref, pd.DataFrame({"ticker": ["HLT"], "sector": ["Gaming"], "daily_vol": [0.031]})])
oops = summary.reset_index().merge(ref_dup, on="ticker", how="left")
print(f"5 tickers in, {len(oops)} rows out — HLT now appears twice:")
oops[oops.ticker == "HLT"]

5 tickers in, 6 rows out — HLT now appears twice:


,ticker,trades_n,total_shares,gross_notional,avg_price,sector,daily_vol
2,HLT,2,50000,3077000.0,61.75,Lodging,0.018
3,HLT,2,50000,3077000.0,61.75,Gaming,0.031


In [25]:
# The guard rails, in order of preference:
# 1. validate= makes the assumption explicit — it RAISES if violated:
try:
    summary.reset_index().merge(ref_dup, on="ticker", how="left", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e)[:100], "...")
# 2. de-duplicate the reference table first, consciously:
ref_fixed = ref_dup.drop_duplicates(subset="ticker", keep="first")
print("after drop_duplicates:", len(summary.reset_index().merge(ref_fixed, on="ticker", how="left")), "rows — correct")

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge ...
after drop_duplicates: 5 rows — correct


## 3 · The harder rows of the translation table

These are the "gray rows" from the slide — each worth seeing once, slowly.

In [28]:
# IF / nested IF  →  np.where / np.select
looked_up["risk_bucket"] = np.where(looked_up.daily_vol > 0.02, "high",
                           np.where(looked_up.daily_vol > 0.017, "medium", "low"))
looked_up[["ticker", "daily_vol", "risk_bucket"]]

,ticker,daily_vol,risk_bucket
0,CHH,0.019,medium
1,H,0.022,high
2,HLT,0.018,medium
3,MAR,0.016,low
4,WH,NaN,low


In [30]:
# TRIM / PROPER / SUBSTITUTE chains  →  .str methods, chained
messy = pd.Series(["  hilton WORLDWIDE ", "marriott intl.", "HYATT hotels  "])
print(messy.str.strip().str.title().str.replace(".", "", regex=False).tolist())

['Hilton Worldwide', 'Marriott Intl', 'Hyatt Hotels']


In [32]:
# VLOOKUP on TWO keys (the Excel helper-column trick)  →  merge on a list
px = pd.DataFrame({"ticker": ["HLT", "HLT", "MAR"], "date": ["2026-10-01", "2026-10-02", "2026-10-01"],
                   "close": [61.40, 62.05, 155.20]})
pos = pd.DataFrame({"ticker": ["HLT", "MAR"], "date": ["2026-10-02", "2026-10-01"], "shares": [40_000, 25_000]})
pos.merge(px, on=["ticker", "date"], how="left")        # no helper column, no CONCAT key

,ticker,date,shares,close
0,HLT,2026-10-02,40000,62.05
1,MAR,2026-10-01,25000,155.20


## The takeaways (say them back)
1. A pivot is a `groupby`; the spec is *readable text*, not a drag-and-drop you can't audit.
2. A VLOOKUP is a `merge`; count NaNs after every merge; `validate=` turns a silent Excel bug into a loud Python error.
3. The gray rows are the same ideas with more syntax — the cheat sheet (Appendix B of the design doc) has all of them.